# nanochat exercises
## Yoav Ram

Exercises for [nanochat.ipynb](nanochat.ipynb). Run that notebook first — these all assume
its `checkpoints/nanochat_best.pkl` exists and that `nanochat_model.py` has been generated.

Difficulty: unmarked is a short change to existing code, ★ needs a design decision of your
own, ★★ is a small project.

The model is importable:

```python
from nanochat_model import (init_params, forward, attention_forward, qk_norm,
                            precompute_rope, causal_mask, cross_entropy_loss,
                            load_checkpoint, generate)
from bpe import bpe_encode, bpe_decode, bpe_load
```

**The training loop is not.** `sample_batch`, `train_step`, `validate` and the `optax`
optimiser live in `nanochat.ipynb`'s cells, not in `nanochat_model.py` — every exercise that
trains something needs you to copy those four cells across, along with the corpus at
`checkpoints/train_tokens.npy` and the tokenizer at `checkpoints/bpe_tokenizer_train.pkl`.

Several of these ask you to *measure* rather than to build. Before running them, read the
"how do you decide?" section of `nanochat.ipynb`: a difference you cannot distinguish from
run-to-run noise is not a result. There are two floors there, and the wider one is the bar —
about **0.002 nats** between bit-identical re-runs, and about **0.005** across seeds.

### Exercise 1: Scaling experiment
Train models with `d_model ∈ {64, 128, 256}` for 300 steps each. Plot
validation loss vs. parameter count. Does it follow a power law?

### Exercise 2: Replace RMSNorm with LayerNorm
Add a learnable bias $\mathbf{b}$ and mean subtraction:
$$\text{LayerNorm}(\mathbf{x}) = g \cdot \frac{\mathbf{x} - \mu}{\sigma + \epsilon} + b$$
Compare training stability and final loss with RMSNorm.

### Exercise 3: `jax.vmap` for per-sample gradients
Use `jax.vmap(jax.grad(per_sample_loss))` to compute per-example gradients.
How does per-example gradient variance evolve during training?

### Exercise 4: Reproduce the QK-norm A/B at half the budget ★
`nanochat.ipynb` measures unit-L2 against RMS at 4,000 steps. Predict, from the arithmetic in
its attention section, what you expect at **2,000** — then run both and find out whether the
loss gap and the entropy ratio hold at half the budget, or whether the gap is still opening.
Report the seed-to-seed spread alongside the gap; one run of each is not an answer.

### Exercise 5: Temperature calibration ★
Plot the entropy of the output distribution as a function of training step.
At what temperature does the model's output distribution best match the
empirical unigram distribution of TinyStories?

### Exercise 6: Biological sequences ★
Replace TinyStories with a protein sequence dataset (FASTA from UniProt).
The vocabulary is the 20 standard amino acids + gaps. What bits-per-token do you achieve?
What subwords does BPE learn — do they correspond to known motifs?

### Exercise 7: Where does the time go? ★★
Time one `train_step` with `jax.block_until_ready`, then vary `batch_size` and
`context_len` and plot tokens/second against each. Which of the two buys more
throughput, and at what point does the $O(T^2)$ attention term start to show up in
the measurement? Compare what you find against the $C = 6ND$ estimate in `nanochat.ipynb`,
which counts only the matmuls.

*(The vocabulary-size ablation belongs to `bpe-tokenizer.ipynb`, Exercise 1.)*